# Bước 2 — Dịch sách Hán → Việt trên L4 x4, offline

Pipeline dịch Hán → Việt chạy hoàn toàn offline: giữ cách chia đoạn, prompt và resume; mỗi GPU giữ một replica model.

**Chuẩn bị**
1. Chạy `download_dich_model.ipynb` khi Internet On → **Save Version**.
2. Notebook này: **Internet Off**, Accelerator **GPU L4 x4**.
3. *Add Input*: Output model ở bước 1 + Dataset/Output OCR chứa `.txt` chữ Hán.

**Cách chạy:** sửa ô cấu hình → chạy lần lượt. Mỗi GPU giữ một replica model (BF16, SDPA); 4 GPU chia đoạn song song.
Output: `/kaggle/working/dich/<tên sách>.txt` + `dich.zip`; raw JSONL trong `/kaggle/working/dich_raw/` để resume.

**Resume:** Add Input Output của phiên trước, notebook tự tìm `dich_raw/dich_shard*.jsonl`; đoạn đã có bản dịch không rỗng sẽ bỏ qua.
Không cần `pip install`, không cần repo, không cần `HF_TOKEN` khi chạy offline.


In [ ]:
# ===== CẤU HÌNH =====
MODEL_DIR = ""          # "" = tự tìm thư mục có model-manifest.json trong /kaggle/input
INPUT_DIR = ""          # "" = quét mọi .txt trong /kaggle/input; hoặc vd "/kaggle/input/<dataset>/txt"
OUTPUT_BASE = "/kaggle/working"

N_GPU = 4               # L4 x4
GPU_PER_MODEL = 1       # 1 GPU / replica (7B BF16 vừa 24 GB); model lớn hơn có thể đặt 2
BATCH = 16              # số đoạn MỖI GPU mỗi lượt; OOM thì tự chia đôi, hoặc giảm xuống 8
# Tách đoạn tại dòng trống, không giới hạn 300 ký tự.
MAX_NEW_TOKENS = 1200
TEMPERATURE = 0         # 0 = greedy, ổn định; 0.3-0.7 = đa dạng hơn
REPETITION_PENALTY = 1.05
MIN_HAN = 50            # bỏ file ít chữ Hán
BOOKS = []              # [] = tất cả; hoặc glob tên file, vd ["*kinh*"]
LIMIT = None            # vd 2 = 2 đoạn MỖI GPU để thử; None = dịch toàn bộ
BUDGET_H = 10.5         # tự dừng trước giới hạn phiên Kaggle
RESUME_FROM = []        # thêm đường dẫn Output cũ nếu không muốn quét toàn bộ /kaggle/input

SYSTEM = "Bạn là dịch giả chuyên dịch sách Y học cổ truyền từ chữ Hán (văn ngôn và bạch thoại) sang tiếng Việt."
USER = ("Văn bản chữ Hán:\n{text}\n\n"
        "Dịch toàn bộ văn bản trên sang tiếng Việt, đầy đủ ý, văn phong tự nhiên. Tên huyệt, vị thuốc, bài thuốc, "
        "bệnh danh giữ theo âm Hán-Việt (vd: 足三里 -> Túc tam lý). Chỉ viết tiếng Việt: không chép lại chữ Hán, "
        "không dịch sang tiếng Anh, không lời dẫn, không giải thích.")


## 0. Kiểm tra môi trường offline + tìm model

Không cài package. Nếu thiếu `transformers`/`accelerate`, hãy chuẩn bị dependency trong một session online (Dependency Manager) rồi mở session L4 mới.


In [ ]:
import os, sys, json, importlib.metadata as md
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for name in ("torch", "transformers", "accelerate", "huggingface-hub"):
    try:
        print(f"{name}: {md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name}: CHƯA CÀI")

try:
    import torch, transformers
except ImportError as exc:
    raise RuntimeError("Thiếu torch/transformers. Chuẩn bị dependency khi online rồi mở session L4 mới.") from exc

print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA không khả dụng: chọn Accelerator GPU L4 x4."
assert torch.cuda.device_count() >= N_GPU, f"Cần {N_GPU} GPU, chỉ thấy {torch.cuda.device_count()}."
assert all(torch.cuda.get_device_capability(i)[0] >= 8 for i in range(N_GPU)), "Cần GPU Ampere trở lên (L4), không dùng T4."
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | BF16: {torch.cuda.is_bf16_supported()}")


def find_model_dir(value: str) -> Path:
    if value:
        root = Path(value)
        if root.is_file():
            root = root.parent
        if (root / "config.json").is_file():
            return root
        found = sorted({p.parent for p in root.rglob("config.json")
                        if (p.parent / "tokenizer_config.json").is_file()})
        if len(found) != 1:
            raise RuntimeError(f"Tìm thấy {len(found)} thư mục model trong {root}: {found}")
        return found[0]
    found = sorted({p.parent for p in Path("/kaggle/input").rglob("model-manifest.json")})
    if len(found) == 1:
        return found[0]
    if not found:
        found = sorted({p.parent for p in Path("/kaggle/input").rglob("config.json")
                        if (p.parent / "tokenizer_config.json").is_file()
                        and (list(p.parent.glob("*.safetensors")) or list(p.parent.glob("*.bin")))})
    if len(found) != 1:
        raise RuntimeError(f"Cần đúng 1 model trong /kaggle/input, thấy {len(found)}: {found}. "
                           "Điền MODEL_DIR trực tiếp tới thư mục chứa config.json.")
    return found[0]


model_root = find_model_dir(MODEL_DIR)
assert (model_root / "config.json").is_file(), f"Thiếu config.json trong {model_root}"
assert (model_root / "tokenizer_config.json").is_file(), f"Thiếu tokenizer_config.json trong {model_root}"
assert (model_root / "tokenizer.json").is_file() or (model_root / "tokenizer.model").is_file(), \
    f"Thiếu tokenizer trong {model_root}; tải lại model khi Internet On rồi Add Input Output mới."
assert list(model_root.glob("*.safetensors")) or list(model_root.glob("*.bin")), f"Thiếu trọng số trong {model_root}"
manifest_path = model_root / "model-manifest.json"
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("Model:", manifest.get("model_id"), "| Revision:", manifest.get("revision"))
print("Model directory:", model_root)
print("Tokenizer files:", sorted(p.name for p in model_root.glob("*token*")))


## 1. Worker dịch (không cần sửa)

Mỗi tiến trình nhận 1 GPU qua `CUDA_VISIBLE_DEVICES`, nạp model **local_files_only**, dùng BF16 + SDPA.
Gặp CUDA OOM sẽ tự chia đôi batch. Kết quả ghi/flush từng đoạn vào `dich_shard<k>.jsonl`.


In [ ]:
%%writefile /kaggle/working/dich_worker.py
"""Dịch các đoạn Hán -> Việt bằng 1 LLM (transformers); chạy lại thì bỏ qua đoạn đã dịch.

Notebook L4 x4 chạy 4 tiến trình, mỗi tiến trình 1 GPU (CUDA_VISIBLE_DEVICES).
"""
from __future__ import annotations

import argparse
import sys
import traceback
import json
import time
from pathlib import Path


def load_done(out: Path) -> set[str]:
    done = set()
    for f in out.glob("dich_shard*.jsonl"):
        for line in open(f, encoding="utf-8"):
            try:
                row = json.loads(line)
                if row.get("vi", "").strip():
                    done.add(row["id"])
            except (json.JSONDecodeError, KeyError, TypeError):
                pass
    return done


def build_prompt(tok, system: str, user: str) -> str:
    """Prompt theo chat template của model (model không nhận vai system thì gộp vào user)."""
    for msgs in ([{"role": "system", "content": system}, {"role": "user", "content": user}],
                 [{"role": "user", "content": system + "\n\n" + user}]):
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)  # Qwen3: tắt chế độ suy nghĩ
        except Exception:  # noqa: BLE001
            continue
    raise RuntimeError("Model không có chat template -- hãy chọn bản *-Instruct / *-Chat")


MODEL_CACHE = {}  # giữ tokenizer/model trong tiến trình giữa các lượt dịch

def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--chunks", required=True, help='jsonl, mỗi dòng {"id", "zh"}')
    ap.add_argument("--prompt", required=True, help='json {"system", "user"}; user chứa {text}')
    ap.add_argument("--out", required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshard", type=int, default=1)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--max-new", type=int, default=1200)
    ap.add_argument("--temperature", type=float, default=0.0, help="0 = luôn chọn từ chắc nhất")
    ap.add_argument("--rep-penalty", type=float, default=1.05)
    ap.add_argument("--limit", type=int, default=None, help="chỉ dịch N đoạn (chạy thử)")
    a = ap.parse_args()

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    chunks = [json.loads(l) for l in open(a.chunks, encoding="utf-8") if l.strip()]
    done = load_done(out)
    todo = [c for i, c in enumerate(chunks) if i % a.nshard == a.shard and c["id"] not in done][:a.limit]
    print(f"[dich#{a.shard}] còn {len(todo)} đoạn", flush=True)
    if not todo:
        return

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    cached = MODEL_CACHE.get(a.model)
    tok = cached[0] if cached else AutoTokenizer.from_pretrained(a.model, local_files_only=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    use_cuda = torch.cuda.is_available()
    if use_cuda:
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        kw = {"dtype": dtype, "device_map": "auto", "attn_implementation": "sdpa"}
    else:
        dtype = torch.float32
        kw = {"dtype": dtype}
    if cached:
        model = cached[1]
        print(f'[dich#{a.shard}] tái sử dụng model trên GPU', flush=True)
    else:
        print(f'[dich#{a.shard}] bắt đầu nạp model (VRAM có thể được cấp phát trước khi xong)', flush=True)
        try:
            model = AutoModelForCausalLM.from_pretrained(a.model, local_files_only=True, **kw).eval()
        except TypeError:  # transformers cũ chỉ nhận torch_dtype
            kw['torch_dtype'] = kw.pop('dtype')
            model = AutoModelForCausalLM.from_pretrained(a.model, local_files_only=True, **kw).eval()
        MODEL_CACHE[a.model] = (tok, model)
    print(f"[dich#{a.shard}] model={a.model} device={model.device} dtype={model.dtype} "
          f"batch={a.batch}", flush=True)

    pr = json.loads(Path(a.prompt).read_text(encoding="utf-8"))

    def decode_batch(batch):
        prompts = [build_prompt(tok, pr["system"], pr["user"].replace("{text}", c["zh"])) for c in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            sample = ({"do_sample": True, "temperature": a.temperature, "top_p": 0.95}
                      if a.temperature > 0 else {"do_sample": False})
            gen = model.generate(**enc, max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty,
                                 pad_token_id=tok.pad_token_id, **sample)
        gen = gen[:, enc["input_ids"].shape[1]:]
        rows = []
        for c, g in zip(batch, gen):
            n_tok = int((g != tok.pad_token_id).sum())
            rows.append({"id": c["id"], "vi": tok.decode(g, skip_special_tokens=True).strip(),
                         "hit_max": n_tok >= a.max_new})
        return rows

    def run_batch(batch):
        """Gặp CUDA OOM thì chia đôi batch và thử lại, thay vì dừng cả shard."""
        try:
            return decode_batch(batch)
        except torch.cuda.OutOfMemoryError:
            pass  # thoát except trước khi thử lại để không giữ tensor qua traceback
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        if len(batch) == 1:
            raise torch.cuda.OutOfMemoryError(
                f'[dich#{a.shard}] vẫn OOM với 1 đoạn sau khi giải phóng batch lỗi.')
        mid = len(batch) // 2
        print(f"   OOM: chia {len(batch)} đoạn thành {mid}+{len(batch) - mid}", flush=True)
        return run_batch(batch[:mid]) + run_batch(batch[mid:])

    todo.sort(key=lambda c: len(c["zh"]), reverse=True)  # đoạn dài gần nhau -> ít padding
    t_start, n_done = time.time(), 0
    fout = open(out / f"dich_shard{a.shard}.jsonl", "a", encoding="utf-8")
    for b in range(0, len(todo), a.batch):
        batch = todo[b:b + a.batch]
        rows = run_batch(batch)
        for row in rows:
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
        fout.flush()
        n_done += len(batch)
        eta = (time.time() - t_start) / n_done * (len(todo) - n_done) / 60
        print(f"[dich#{a.shard}] {n_done}/{len(todo)} đoạn | còn ~{eta:.0f} phút\n"
              f"   ZH: {batch[-1]['zh'][:60]}\n   VI: {rows[-1]['vi'][:120]}", flush=True)
    fout.close()


if __name__ == "__main__":
    if '--serve' not in sys.argv:
        main()
    else:
        for line in sys.stdin:
            job = json.loads(line)
            sys.argv = [sys.argv[0], *job['args']]
            status = {'ok': True}
            try:
                main()
            except BaseException:
                traceback.print_exc()
                status = {'ok': False}
            marker = Path(job['status'])
            temporary = marker.with_suffix('.tmp')
            temporary.write_text(json.dumps(status), encoding='utf-8')
            temporary.replace(marker)


## Xóa kết quả dịch cũ — chạy trước bước 2
Cell dưới xóa `dich_raw`, `dich`, `dich.zip`, `chunks.jsonl`, `prompt.json` trong `OUTPUT_BASE` và tắt resume từ Add Input. Không xóa model hoặc dữ liệu nguồn. Chạy cell này sẽ mất toàn bộ kết quả dịch hiện tại; bỏ qua cell nếu muốn resume. Muốn bật lại resume sau khi đã chạy, đặt `FRESH_TRANSLATION = False` trước bước 2.


In [ ]:
from pathlib import Path
import shutil

# Dừng worker trước khi xóa để tránh tiến trình cũ còn ghi kết quả.
if 'stop_workers' in globals():
    stop_workers()

work_to_clean = Path(OUTPUT_BASE).resolve()
assert work_to_clean != Path('/'), 'Không được dùng OUTPUT_BASE là thư mục gốc.'
for name in ('dich_raw', 'dich', 'dich.zip', 'chunks.jsonl', 'prompt.json'):
    target = work_to_clean / name
    if target.is_symlink():
        target.unlink()
    elif target.is_dir():
        shutil.rmtree(target)
    elif target.exists():
        target.unlink()
    print('Đã dọn:', target)

FRESH_TRANSLATION = True  # không nạp lại kết quả cũ từ /kaggle/input
RESUME_FROM = []
print('Sẵn sàng dịch mới. Chạy tiếp từ bước 2; model và dữ liệu nguồn được giữ nguyên.')


## 2. Đọc txt → chia đoạn + nạp kết quả cũ

Tách tại một hoặc nhiều dòng trống, giống notebook EraX; giữ xuống dòng đơn bên trong đoạn, không cắt theo số ký tự. Chạy cell xóa kết quả cũ khi đổi cách chia để tránh trộn kết quả resume. Đoạn dài hơn có thể cần giảm BATCH và tăng MAX_NEW_TOKENS.


In [ ]:
import fnmatch, glob, hashlib, json, os, re, shutil, subprocess, time

WORK = Path(OUTPUT_BASE)
RAW, DICH = WORK / "dich_raw", WORK / "dich"
RAW.mkdir(parents=True, exist_ok=True)
DICH.mkdir(parents=True, exist_ok=True)
WORKER = Path("/kaggle/working/dich_worker.py")
HAN = re.compile(r"[㐀-鿿\U00020000-\U0002ffff]")

def make_chunks(text):
    """Tách tại dòng trống; giữ nguyên xuống dòng đơn trong mỗi khối."""
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    return [part.strip() for part in re.split(r'\n[ \t]*\n(?:[ \t]*\n)*', text)
            if part.strip()]

source = Path(INPUT_DIR) if INPUT_DIR else Path("/kaggle/input")
assert source.is_dir(), f"INPUT_DIR không tồn tại: {source}"
books = {}
for p in sorted(source.rglob("*.txt")):
    if BOOKS and not any(fnmatch.fnmatch(p.stem, pattern) for pattern in BOOKS):
        continue
    b = p.stem
    text = open(p, encoding="utf-8", errors="ignore").read()
    if b in books or len(HAN.findall(text)) < MIN_HAN:
        continue  # trùng tên / không phải văn bản chữ Hán
    books[b] = [{"id": f"{b}|{i}|{hashlib.md5(c.encode()).hexdigest()[:8]}", "zh": c}
                for i, c in enumerate(make_chunks(text))]
    print(f"{b:<50} {len(text):8,} ký tự -> {len(books[b]):5d} đoạn")
assert books, "Không thấy file .txt chữ Hán nào trong INPUT_DIR -- đã Add Input chưa?"
ZH = {c["id"]: c["zh"] for cs in books.values() for c in cs}
CHUNKS = str(WORK / "chunks.jsonl")
with open(CHUNKS, "w", encoding="utf-8") as f:
    for cs in books.values():
        f.writelines(json.dumps(c, ensure_ascii=False) + "\n" for c in cs)
json.dump({"system": SYSTEM, "user": USER}, open(WORK / "prompt.json", "w", encoding="utf-8"), ensure_ascii=False)
print(f"Tổng: {len(books)} sách, {len(ZH)} đoạn")

# Resume: tìm shard của các Output cũ (Add Input) và copy vào RAW với tên riêng.
def previous_shards():
    if globals().get('FRESH_TRANSLATION', False):
        return  # lượt dịch mới: không nhập lại shard cũ từ Add Input
    seen = set()
    for folder in [Path("/kaggle/input"), *map(Path, RESUME_FROM)]:
        if not folder.is_dir():
            continue
        for f in sorted(folder.rglob("dich_shard*.jsonl")):
            if f.parent.name != "dich_raw" or f.resolve() in seen:
                continue
            seen.add(f.resolve())
            yield f

for f in previous_shards():
    tag = hashlib.md5(str(f).encode()).hexdigest()[:8]
    dst = RAW / f"dich_shard_prev{tag}.jsonl"
    if not dst.exists():
        shutil.copy(f, dst); print("Nạp lại", f)

# Giữ pool khi chạy lại ô này; chỉ stop_workers() mới giải phóng model.
_DICH_POOL = globals().get('_DICH_POOL', {})

def stop_workers():
    for _, p in _DICH_POOL.values():
        if p.poll() is None:
            p.terminate()
    for _, p in _DICH_POOL.values():
        try:
            p.wait(timeout=10)
        except subprocess.TimeoutExpired:
            p.kill()
            p.wait()
        if p.stdin:
            p.stdin.close()
    _DICH_POOL.clear()
    print('Đã dừng worker và giải phóng model trên GPU.')

def run_dich(limit=None, budget_h=None):
    """N_GPU / GPU_PER_MODEL replica chạy song song, chia đều các đoạn."""
    assert WORKER.is_file(), f"Thiếu {WORKER}; chạy lại ô %%writefile."
    import uuid
    budget_h = BUDGET_H if budget_h is None else budget_h
    assert GPU_PER_MODEL > 0 and N_GPU % GPU_PER_MODEL == 0
    n = N_GPU // GPU_PER_MODEL
    pool_key = (str(model_root.resolve()), N_GPU, GPU_PER_MODEL, str(WORKER), WORKER.stat().st_mtime_ns)
    if _DICH_POOL and (set(_DICH_POOL) != {(pool_key, k) for k in range(n)}
                       or any(p.poll() is not None for _, p in _DICH_POOL.values())):
        stop_workers()
    jobs = []
    for k in range(n):
        gpus = ",".join(str(g) for g in range(k * GPU_PER_MODEL, (k + 1) * GPU_PER_MODEL))
        cmd = [sys.executable, str(WORKER), "--model", str(model_root), "--chunks", CHUNKS,
               "--prompt", str(WORK / "prompt.json"), "--out", str(RAW), "--shard", str(k), "--nshard", str(n),
               "--batch", str(BATCH), "--max-new", str(MAX_NEW_TOKENS), "--temperature", str(TEMPERATURE),
               "--rep-penalty", str(REPETITION_PENALTY)]
        if limit is not None:
            cmd += ["--limit", str(limit)]
        log = RAW / f"_dich_{k}.log"
        key = (pool_key, k)
        if key not in _DICH_POOL:
            with open(log, 'a') as stream:
                p = subprocess.Popen([sys.executable, str(WORKER), '--serve'],
                    stdin=subprocess.PIPE, text=True, stdout=stream, stderr=subprocess.STDOUT,
                    env={**os.environ, 'CUDA_VISIBLE_DEVICES': gpus, 'PYTHONUNBUFFERED': '1'})
            _DICH_POOL[key] = (log, p)
        _, p = _DICH_POOL[key]
        marker = RAW / f'_status_{uuid.uuid4().hex}.json'
        p.stdin.write(json.dumps({'args': cmd[2:], 'status': str(marker)}) + '\n')
        p.stdin.flush()
        jobs.append((log, p, marker))
    print(f"Chạy {n} worker: " + ", ".join(f"GPU {k * GPU_PER_MODEL}-{(k + 1) * GPU_PER_MODEL - 1}" for k in range(n)),
          flush=True)
    t0 = t_print = time.time()
    try:
        while any(not marker.exists() and p.poll() is None for _, p, marker in jobs):
            time.sleep(2)
            if time.time() - t0 > budget_h * 3600:
                print(f'!!! Hết {budget_h} giờ; dừng worker, giữ kết quả đã ghi.')
                stop_workers()
                break
            if time.time() - t_print > 120:
                t_print = time.time()
                last = [(l.read_text(errors='ignore').strip().splitlines() or ['...'])[-3:]
                        for l, _, _ in jobs]
                print(f'[{(time.time() - t0) / 60:5.1f} phút]\n' + '\n'.join('\n'.join(x) for x in last), flush=True)
    except BaseException:
        stop_workers()  # tránh lượt cũ còn ghi file khi chạy lại sau Interrupt
        raise
    failed = False
    for log, p, marker in jobs:
        ok = marker.exists() and json.loads(marker.read_text()).get('ok', False)
        marker.unlink(missing_ok=True)
        if not ok:
            failed = True
            print(f"\n!!! {log.name} lỗi, 20 dòng cuối log:")
            print("\n".join(open(log, errors="ignore").read().splitlines()[-20:]))
    if failed:
        print("\nKiểm tra log rồi chạy lại; các đoạn đã dịch sẽ được giữ nguyên.")
        stop_workers()
    else:
        print('Xong lượt dịch. Model vẫn trên GPU; lượt tiếp theo không nạp lại.')

def load_dich():
    out = {}
    for f in RAW.glob("dich_shard*.jsonl"):
        for line in open(f, encoding="utf-8"):
            try:
                r = json.loads(line)
                if r.get("vi", "").strip():
                    out[r["id"]] = r
            except (json.JSONDecodeError, KeyError, TypeError):
                pass
    return out


In [ ]:
import shutil
import time

src = model_root
dst = Path("/kaggle/working/model_local")
marker = dst / ".copy_complete"

stop_workers() if "stop_workers" in globals() else None

t0 = time.time()
if not marker.exists():
    required = sum(p.stat().st_size for p in src.rglob("*") if p.is_file())
    free = shutil.disk_usage(dst.parent).free
    assert free > required * 1.1, "Không đủ dung lượng để copy model"

    print("Đang copy model sang /kaggle/working...", flush=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    marker.touch()

model_root = dst
print(f"Model local: {model_root} | {time.time() - t0:.1f} giây")

## 3. Dịch thử vài đoạn

`LIMIT` tính theo **mỗi GPU**: `limit=2` với 4 GPU dịch tối đa 8 đoạn. Xem bản dịch trước khi chạy toàn bộ.

Worker giữ model trên GPU sau lượt thử. Chạy ô 4 trong cùng kernel/session sẽ tái sử dụng, không nạp lại. VRAM còn chiếm nhưng GPU 0% khi chờ là bình thường. Lần nạp đầu có thể đã cấp phát VRAM dù thanh Loading weights chưa tăng.
Gọi `stop_workers()` để chủ động giải phóng GPU. Interrupt, hết budget, lỗi worker hoặc đổi model/cấu hình GPU sẽ dừng pool; lần sau phải nạp lại. Restart kernel/session không giữ được model. Nếu sửa ô worker, chạy lại ô %%writefile trước khi dịch.


In [ ]:
run_dich(limit=2, budget_h=1)
for r in list(load_dich().values())[:8]:
    print(f"ZH: {ZH.get(r['id'], '')[:200]}\nVI: {r['vi'][:400]}\n")


## 4. Dịch toàn bộ

Có thể dừng session bất cứ lúc nào; lần sau Add Input Output này để resume. Notebook tự dừng sau `BUDGET_H` giờ.


In [ ]:
run_dich(limit=LIMIT)


## 5. Ghi ra txt + zip

Mỗi sách → một file UTF-8; đoạn chưa dịch ghi `[chưa dịch]`.


In [ ]:
done = load_dich()
for b, cs in books.items():
    vi = [done[c["id"]]["vi"] if c["id"] in done else "[chưa dịch]" for c in cs]
    (DICH / f"{b}.txt").write_text("\n\n".join(vi) + "\n", encoding="utf-8")
    n_ok = sum(c["id"] in done for c in cs)
    n_max = sum(done[c["id"]].get("hit_max", False) for c in cs if c["id"] in done)
    print(f"{b:<50} {n_ok}/{len(cs)} đoạn" + (f", {n_max} đoạn chạm MAX_NEW_TOKENS (có thể bị cụt)" if n_max else ""))
shutil.make_archive(str(WORK / "dich"), "zip", DICH)
print("\nKết quả:", DICH / "*.txt", "và", WORK / "dich.zip")
